In [ ]:
from google.colab import auth
import pandas as pd

# Autenticación de tu cuenta de Google
auth.authenticate_user()

# Configuración del acceso al Bucket
BUCKET_NAME = "coreproyecto-ds-datos"
RAW_DATA_PATH = f"gs://{BUCKET_NAME}/raw/"

print("✅ Conectado exitosamente al almacenamiento de easyMoney")

In [ ]:
!gsutil ls gs://coreproyecto-ds-datos/raw/

# SOCIODEMOGRÁFICO

In [ ]:
import pandas as pd

BASE = "gs://coreproyecto-ds-datos/raw/"
df_soc = pd.read_csv(BASE + "customer_sociodemographics.csv")

print("SHAPE:", df_soc.shape)
display(df_soc.head())

In [ ]:
df_soc.info()

In [ ]:
df_soc.isna().sum().sort_values(ascending=False)

In [ ]:
df_soc.duplicated().sum()


In [ ]:
df_soc.duplicated(subset=["pk_cid", "pk_partition"]).sum()

Limpieza Columnas

In [ ]:
# Eliminar columna Unnamed: 0
df_soc = df_soc.drop(columns=["Unnamed: 0"])

PK_partition

In [ ]:
# Convertir pk_partition a datetime (Año-mes)
df_soc["pk_partition"] =pd.to_datetime(df_soc["pk_partition"])

In [ ]:
df_soc["pk_partition"].head()

Region Code

In [ ]:
# Convertir region_code a int
df_soc["region_code"] = df_soc["region_code"].astype("Int64")

In [ ]:
# Calculando el total de filas, las filas que son == España, y restando Total filas - Filas España para ver si el resto coincide con el número de nulos de region_id (2252 nulos)
total_rows = df_soc.shape[0]
es_rows = df_soc[df_soc["country_id"] == "ES"].shape[0]

print("Total filas:", total_rows)
print("Filas España:", es_rows)
print("No España:", total_rows - es_rows)

Los nulos en "region code" son países que no son España (de momento dejamos como nulos, aunque se puede poner -1 o 0 pero para power BI tal vez sea peor)

Fallecidos

In [ ]:
df_soc["deceased"].unique()

In [ ]:
# Como es una binaria lógica (si/no), ponemos 1 si está fallecido y 0 si no lo está:
df_soc["deceased"] = df_soc["deceased"].map({"N": 0, "S": 1})
df_soc["deceased"] = df_soc["deceased"].astype(int)

In [ ]:
df_soc["deceased"].value_counts()

Género

In [ ]:
df_soc["gender"].unique()

In [ ]:
# Imputamos los nulos como Unknow:
df_soc["gender"] = df_soc["gender"].fillna("Unknown")

In [ ]:
# Para una mejor visualización en power BI renombramos H con Hombre y V con Mujer:
df_soc["gender"] = df_soc["gender"].replace({
    "H": "Male",
    "V": "Female"
})

In [ ]:
df_soc["gender"].value_counts(normalize=True)

Edad

In [ ]:
df_soc["age"].describe()

In [ ]:
# Mirar cuántas edades mayor a 100 hay, por si hubiera algún error:
df_soc[df_soc["age"] > 100]

Se detectan 201 registros con edades superiores a 100 años. Aunque no se descarta su veracidad, representan un porcentaje marginal de la muestra y constituyen valores extremos desde el punto de vista estadístico y de negocio, especialmente en el contexto financiero como es esta empresa. Por lo que:
- Se podría capear a 100 años para estabilizar la distribución y mantener la coherencia del negocio.
- Crear una nueva variable "age_erderly" para agrupar a aquellas personas > 95 o 100 años.
- Dejarlo tal cual si vemos que estos outliers no afectarán al modelo.


Salario

In [ ]:
df_soc["salary"].describe()

In [ ]:
df_soc[df_soc["salary"] > 1_000_000].shape

8418 personas cobran más de 1.000.000 euros (0,2% aprox) Puede que muchas sean empresas


In [ ]:
df_soc["salary"].isna().sum()

In [ ]:
df_soc.isna().sum().sort_values()

In [ ]:
df_soc.head()

In [ ]:
df_soc.info()

Se han mantenido los valores nulos en "salary" y "region_code" de forma intencionada.

En salary, el nulo indica que el cliente no ha informado sus ingresos, por lo que imputar 0 o la media generaría sesgo en el análisis.

En region_code, los nulos corresponden principalmente a clientes no residentes en España, donde la región no aplica.

# CUSTOMER PRODUCTS

In [ ]:
df_prod = pd.read_csv(BASE + "customer_products.csv")

print("SHAPE:", df_prod.shape)
display(df_prod.head())

In [ ]:
df_prod.info()

In [ ]:
# comprobando nulos:
df_prod.isna().sum().sort_values(ascending=False)

In [ ]:
df_prod.duplicated().sum()

In [ ]:
# comprobando si algun usuario está duplicado en un mes en concreto
df_prod.duplicated(subset=["pk_cid", "pk_partition"]).sum()

In [ ]:
df_prod.describe()

Limpieza de columnas

In [ ]:
# Eliminar columna Unnamed : 0
df_prod = df_prod.drop(columns=["Unnamed: 0"])

PK_partition a DateTime

In [ ]:
# Convertir pk_partition a fecha
df_prod["pk_partition"] = pd.to_datetime(df_prod["pk_partition"])

Payroll y pension_plan

In [ ]:
# Rellenar nulos en payroll y pension_plan con 0 (Son columnas binarias, que deberían ser int, pero son float porque como tienen 61 nulos Pandas los convierte en float)
df_prod["payroll"] = df_prod["payroll"].fillna(0)
df_prod["pension_plan"] = df_prod["pension_plan"].fillna(0)

In [ ]:
# Convertimos a int
df_prod["payroll"] = df_prod["payroll"].astype(int)
df_prod["pension_plan"] = df_prod["pension_plan"].astype(int)

In [ ]:
df_prod.info()

In [ ]:
df_prod.describe()

In [ ]:
df_prod.head()

In [ ]:
df_prod.isna().sum().sort_values(ascending=False)

In [ ]:
# Pasando los DF sociodemoraphics y customer_products a CSV
df_soc.to_csv("bi_customer_sociodemographics.csv", index=False)
df_prod.to_csv("bi_customer_products.csv", index=False)

In [ ]:
!gsutil cp bi_customer_sociodemographics.csv gs://coreproyecto-ds-datos/procesado/
!gsutil cp bi_customer_products.csv gs://coreproyecto-ds-datos/procesado/

# SALES

In [ ]:
BUCKET_NAME = "coreproyecto-ds-datos"
FILE_NAME = "sales.csv"  # Cambia aquí el que quieras
RAW_DATA_PATH = f"gs://{BUCKET_NAME}/raw/{FILE_NAME}"

df = pd.read_csv(RAW_DATA_PATH)

df.head()

In [ ]:
# Eliminamos la columna de índice residual
df_sales = df.drop(columns=['Unnamed: 0'], errors='ignore')

# Limpiamos posibles espacios en blanco en los nombres de las columnas
df_sales.columns = df_sales.columns.str.strip()

# Renombramos el ID
df_sales = df_sales.rename(columns={'product_ID': 'product_id'})

# Visualizamos el resultado
df_sales.head()

In [ ]:
# Convertimos month_sale a formato datetime
df_sales['month_sale'] = pd.to_datetime(df_sales['month_sale'])

# Verificamos que ahora sea datetime64[ns]
print(df_sales['month_sale'].dtype)

In [ ]:
# Aseguramos que net_margin sea numérico
df_sales['net_margin'] = pd.to_numeric(df_sales['net_margin'], errors='coerce')

# Llenar nulos con 0
df_sales['net_margin'] = df_sales['net_margin'].fillna(0)

# Redondear a 2 decimales
df_sales['net_margin'] = df_sales['net_margin'].round(2)

In [ ]:
# Comprobamos si hay filas de ventas exactamente iguales
duplicados_totales = df_sales.duplicated().sum()
print(f"Filas duplicadas totales: {duplicados_totales}")

# Comprobamos si hay IDs de venta repetidos
id_duplicados = df_sales['cid'].duplicated().sum()
print(f"IDs de venta (cid) duplicados: {id_duplicados}")


In [ ]:
# Resumen
print(df_sales.describe())

# Verificamos que no queden valores nulos en ninguna columna
print("\nNulos por columna:")
print(df_sales.isnull().sum())

In [ ]:
df_sales

# PRODUCT_DESCRIPTION



In [ ]:
BUCKET_NAME = "coreproyecto-ds-datos"
FILE_NAME = "product_description.csv"  # Cambia aquí el que quieras
RAW_DATA_PATH = f"gs://{BUCKET_NAME}/raw/{FILE_NAME}"

df = pd.read_csv(RAW_DATA_PATH)

df.head()

In [ ]:
# Quitamos la columna innecesaria
df_products= df.drop(columns=['Unnamed: 0'], errors='ignore')

# Limpiamos espacios en los nombres de columnas
df_products.columns = df_products.columns.str.strip()

# Renombramos el ID para que coincida exactamente con el de ventas
df_products = df_products.rename(columns={'pk_product_ID': 'product_id'})

df_products.head()

In [ ]:
# Pasamos todo a minúsculas y quitamos espacios al principio/final de cada celda
df_products['product_desc'] = df_products['product_desc'].str.lower().str.strip()
df_products['family_product'] = df_products['family_product'].str.lower().str.strip()

# Reemplazamos guiones bajos por espacios para que se vea mejor
df_products['product_desc'] = df_products['product_desc'].str.replace('_', ' ')
df_products['family_product'] = df_products['family_product'].str.replace('_', ' ')

print("Texto normalizado.")
df_products.head()

In [ ]:
# Comprobar si faltan nombres de productos o familias
print("Nulos por columna:")
print(df_products.isnull().sum())

# Ver cuántos productos únicos tenemos por familia
print("\nDistribución por familia de producto:")
print(df_products['family_product'].value_counts())

In [ ]:
df_products

# CUSTOMER COMERCIAL ACTIVITY

In [ ]:

BASE = "gs://coreproyecto-ds-datos/raw/"
df_commercial = pd.read_csv(BASE + "customer_commercial_activity.csv", index_col=0)

Miramos la estructura del DF

In [ ]:
df_commercial.shape

In [ ]:
df_commercial.info()

In [ ]:
df_commercial.head()

In [ ]:
df_commercial.describe()

In [ ]:
df_commercial.describe(include = "object")

Data Cleaning.

In [ ]:
#Cambiamos a datetime las columnas "pk_partition" y "entry_date"
df_commercial[['pk_partition', 'entry_date']] = df_commercial[['pk_partition', 'entry_date']].apply(pd.to_datetime, format="%Y-%m")

In [ ]:
#Visualizamos los nulos

In [ ]:
df_commercial.isnull().sum()

Revisamos los nulos en entry_channel: Al analizar el dataset, detectamos un 133033 (2.23%) de valores nulos. Tras una inspección, observamos que la mayoría de estos nulos ocurren en el mes de entrada del cliente

In [ ]:
# 1. Identificar los pk_cid que tienen "mezcla" de nulos y valores
mixed_segment_ids = (
    df_commercial.groupby("pk_cid")["segment"]
    .agg(
        has_null=lambda x: x.isna().any(),
        has_value=lambda x: x.notna().any()
    )
    .query("has_null & has_value")
    .reset_index()[["pk_cid"]]
)

# 2. Filtrar el DataFrame original para ver el detalle de esos clientes
detalle_mezclados = df_commercial.merge(mixed_segment_ids, on="pk_cid").sort_values(["pk_cid", "pk_partition"])

print(f"Total de clientes con datos inconsistentes: {len(mixed_segment_ids)}")
print(detalle_mezclados)

In [ ]:
df_clean = df_commercial.sort_values(["pk_cid", "pk_partition"]).copy()

# Aplicamos backward_fill para que el segmento del mes 2 cubra el nulo del mes 1
# Pero solo dentro del mismo cliente (groupby pk_cid)
df_clean["segment"] = df_clean.groupby("pk_cid")["segment"].transform(lambda x: x.bfill())

# Verificamos si redujimos los nulos
print(f"Nulos antes: {df_commercial['segment'].isna().sum()}")
print(f"Nulos después: {df_clean['segment'].isna().sum()}")

In [ ]:
# Imputamos los nulos restantes como '00 - NO SEGMENTADO'
df_clean["segment"] = df_clean["segment"].fillna("00 - NO SEGMENTADO")

# Verificación final
print(f"Nulos finales en segment: {df_clean['segment'].isna().sum()}")

Para la columna de "entry_channel", decidimos es aplicar un Forward Fill y luego un Backward Fill dentro de cada grupo de cliente.

In [ ]:
# Agrupamos por cliente y vemos si tiene nulos y no nulos
check_recuperables = df_clean.groupby('pk_cid')['entry_channel'].agg(
    tiene_nulos = lambda x: x.isnull().any(),
    tiene_valores = lambda x: x.notnull().any()
)

# Filtramos los que cumplen ambas condiciones
recuperables = check_recuperables[check_recuperables['tiene_nulos'] & check_recuperables['tiene_valores']]

print(f"Clientes con canales recuperables: {len(recuperables)}")

In [ ]:
# Ordenamos para asegurar que el llenado siga la línea de tiempo
df_clean = df_clean.sort_values(['pk_cid', 'pk_partition'])

# Aplicamos ffill y bfill agrupado por cliente
# Esto rellena los huecos usando cualquier valor disponible en la historia del pk_cid
df_clean['entry_channel'] = df_clean.groupby('pk_cid')['entry_channel'].ffill()
df_clean['entry_channel'] = df_clean.groupby('pk_cid')['entry_channel'].bfill()

# Verificación del nuevo estado de nulos
nulos_finales = df_clean['entry_channel'].isnull().mean() * 100
print(f"Nuevo porcentaje de nulos en entry_channel: {nulos_finales:.4f}%")

In [ ]:
#Imputamos los nulos restantes con "UNKNOWN":
df_clean['entry_channel'] = df_clean['entry_channel'].fillna('UNKNOWN')

In [ ]:
df_clean.isnull().sum()

# UNIR TABLAS PARA POWER BI

**1.** **Dataset cliente**

**2.** **Dataset Sales**

**3.** **Dataset product description**

**1. DATASET CLIENTE**

- Creamos nuevo "df_customer" agrupando los .csv sociodemographics, customer_products, customer_commercial_activity para importar a POwerBI:

In [ ]:
df_sales[~df_sales["customer_month_key"].isin(df_customer["pk_cid"])].shape

In [ ]:
df_customer = df_soc.merge(
    df_clean,
    on=["pk_cid", "pk_partition"],
    how="left"
).merge(
    df_prod,
    on=["pk_cid", "pk_partition"],
    how="left"
 )

In [ ]:
df_customer

- Comprobamos que no hay duplicados en "df_customer":

In [ ]:
df_customer.duplicated(subset=["pk_cid", "pk_partition"]).sum()

**2. DATASET SALES**

- El df "Sales" es el df principal. No necesita modificación para importar a PowerBI

In [ ]:
df_sales

**3. DATASET PRODUCT DESCRIPTION**

- Renombramos la variable "pk_product_ID" por "product_ID" para que sea posible la relación con df_sales para POwerBI:

In [ ]:
df_products.rename(columns={"pk_product_ID": "product_ID"}, inplace=True)

In [ ]:
df_products.head()

- Creamos clave compuesta para poder relacionar los ID de df_sales y df_customer: df_sales["customer_month_key"] = (


In [ ]:
df_sales["customer_month_key"] = (
    df_sales["cid"].astype(str) + "_" +
    df_sales["month_sale"].astype(str)
 )

df_customer["customer_month_key"] = (
    df_customer["pk_cid"].astype(str) + "_" +
    df_customer["pk_partition"].astype(str)
 )

# COMPROBANDO PORQUE DA ERROR CUSTOMER Y SALES EN POWER BI

In [ ]:
sales_keys = set(df_sales["customer_month_key"])
customer_keys = set(df_customer["customer_month_key"])

len(sales_keys & customer_keys), len(sales_keys), len(customer_keys)

In [ ]:
df_customer[["gender", "age", "salary"]].isnull().sum()

In [ ]:
df_customer_sales = df_customer[
    df_customer["customer_month_key"].isin(df_sales["customer_month_key"])
]

df_customer_sales[["gender", "age", "salary"]].isnull().sum()

In [ ]:
df_customer[df_customer["customer_month_key"].duplicated(keep=False)] \
    .sort_values("customer_month_key") \
    .head(20)

In [ ]:
missing_keys = list(sales_keys - customer_keys)[:20]
missing_keys

In [ ]:
missing_keys = list(customer_keys - sales_keys)[:20]
missing_keys

In [ ]:
df_customer.groupby("pk_cid")[["gender", "age", "country_id"]].nunique().sort_values(by="gender", ascending=False).head()

In [ ]:
#df_sales["month_sale"].dtype

In [ ]:
#df_sales["month_sale"].head()

In [ ]:
#df_customer["pk_partition"].dtype

In [ ]:
#df_customer["pk_partition"].head()

Con esto concluimos que:

- Df_sales y df_customer se relacionan mediante "customer_month_key".

- Df_sales y df_products se relacionan mediante ·product_id".

In [ ]:
df_sales.nunique()

In [ ]:
df_sales.info()

# Creando df_customer_soc especifico para power BI

In [ ]:
df_soc["pk_cid"].duplicated().sum()

In [ ]:
df_soc.head()

In [ ]:
df_customer_soc = (
    df_soc
    .sort_values("pk_partition")
    .groupby("pk_cid", as_index=False)
    .agg({
        "gender": "first",
        "age": "first",
        "salary": "first",
        "country_id": "first",
        "region_code": "first",
        "deceased": "max"
    })
)

In [ ]:
df_customer_soc["pk_cid"].duplicated().sum()

In [ ]:
 df_sales["cid"].isin(df_customer["pk_cid"]).mean()

# Creando "df_cliente" específico para PowerBI:

Creamos la variable "antiguedad_cliente":

In [ ]:
df_sales['month_sale'] = pd.to_datetime(df_sales['month_sale'])
df_customer['entry_date'] = pd.to_datetime(df_customer['entry_date'])

In [ ]:
df_cliente_ventas = df_sales.merge(
    df_customer[['pk_cid', 'entry_date', 'segment', 'active_customer']],
    left_on='cid',
    right_on='pk_cid',
    how='left'
)

In [ ]:
df_cliente_ventas['antiguedad_cliente'] = (
    (df_cliente_ventas['month_sale'].dt.year - df_cliente_ventas['entry_date'].dt.year) * 12 +
    (df_cliente_ventas['month_sale'].dt.month - df_cliente_ventas['entry_date'].dt.month)
)

In [ ]:
df_cliente = df_cliente_ventas.groupby('pk_cid').agg(
    active_customer=('active_customer', 'first'),
    segment=('segment', 'first'),
    antiguedad_cliente=('antiguedad_cliente', 'max')
).reset_index()

In [ ]:
df_cliente["pk_cid"].duplicated().sum()

In [ ]:
df_cliente = df_customer[['pk_cid','segment','active_customer']].drop_duplicates(subset='pk_cid')
df_cliente.head()

In [ ]:
df_cliente["pk_cid"].duplicated().sum()

In [ ]:
df_cliente.to_csv("df_cliente.csv", index=False)

# CONEXIÓN AZURE MYSQL PARA POWERBI:

In [ ]:
!pip install mysql-connector-python sqlalchemy

In [ ]:
from sqlalchemy import create_engine
import urllib.parse
import pandas as pd
import time # Para medir el rendimiento

# 1. Parámetros
DB_USER = "nuclio"
DB_PASS = "nuclioTFM6"
DB_HOST = "nuclio.mysql.database.azure.com"
DB_PORT = "3306"
DB_NAME = "tfm"

safe_pass = urllib.parse.quote_plus(DB_PASS)
connection_string = f"mysql+mysqlconnector://{DB_USER}:{safe_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

try:
    print("🚀 Iniciando carga MASIVA de datos a Azure...")

    tablas_a_subir = {
        #'df_customer': df_customer,
        #'df_products': df_products,
        #'df_sales_2': df_sales,
        #'df_customer_soc' : df_customer_soc,
        'df_cliente': df_cliente
    }

    for nombre_tabla, df in tablas_a_subir.items():
        start_time = time.time()
        print(f"\n⏳ Subiendo '{nombre_tabla}' ({len(df):,} filas)...")

        # Aumentamos chunksize a 15,000 para reducir el tráfico de red
        df.to_sql(
            name=nombre_tabla,
            con=engine,
            if_exists='replace',
            index=False,
            chunksize=15000,
            method='multi'
        )

        duration = (time.time() - start_time) / 60
        print(f"✅ '{nombre_tabla}' lista en {duration:.2f} minutos.")

    print("\n🎉 ¡Proceso finalizado con éxito!")

except Exception as e:
    print(f"❌ Error: {e}")